# 08 - Collect the RAG documents

We need text to explain the vacancy forecasts, so we collect the official sources listed in
`configs/rag.yaml`: Statistics Finland's job vacancy releases, and the KEHA/ministry monthly
employment bulletins (2013 to now). Everything is saved under `data/raw/rag/`.

In [ ]:
# mount Google Drive and set the working directory to the project path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
except ImportError:
    pass

In [ ]:
# !pip install -q requests beautifulsoup4 pypdf cryptography pandas

import os, re, time, zipfile, io
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pypdf import PdfReader

REPO = Path.cwd()
RAW = REPO / "data" / "raw" / "rag"
TEXT_DIR = RAW / "text"
RAW.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "JobAI-student-project/1.0"}
rows = []  # one row per document: doc_id, title, language, published, url, path

# doria.fi sometimes redirects to a Finnish filename (ä, ö) badly encoded, which
# crashes requests. This fixes the redirect header before requests reads it.
from urllib.parse import quote
session = requests.Session()
session.headers.update(HEADERS)
def _fix_bad_redirect(response, *args, **kwargs):
    location = response.headers.get("Location")
    if location:
        response.headers["Location"] = quote(location.encode("latin-1").decode("latin-1"), safe=":/?&=%#")
    return response
session.hooks["response"].append(_fix_bad_redirect)

## Statistics Finland releases

Three short quarterly releases, in English.

In [ ]:
STATFIN_URLS = [
    "https://stat.fi/en/publication/cmfxzvr0kgj1k07w3ya6x3vac",
    "https://stat.fi/en/publication/cmfxzoyr4gknl07utaizfva9c",
    "https://stat.fi/en/publication/cmfxzk126ghiz07utgwlf7ipu",
]

for url in STATFIN_URLS:
    doc_id = "statfin_" + url.rstrip("/").split("/")[-1]
    html_path = RAW / f"{doc_id}.html"
    if not html_path.is_file():
        response = session.get(url, headers=HEADERS, timeout=60)
        html_path.write_text(response.text, encoding="utf-8")
        time.sleep(1)

    soup = BeautifulSoup(html_path.read_text(encoding="utf-8"), "html.parser")
    title = soup.title.string.split("|")[0].strip() if soup.title else doc_id
    text = soup.get_text(" ", strip=True)
    (TEXT_DIR / f"{doc_id}.txt").write_text(text, encoding="utf-8")

    rows.append({"doc_id": doc_id, "title": title, "language": "en",
                "published": None, "source": "statfin", "url": url})

print(len(STATFIN_URLS), "Statistics Finland releases saved")

## KEHA employment bulletins (2025 onward)

Monthly bulletins, English and Finnish, listed on Työmarkkinatori as permanent `urn.fi` links.

In [ ]:
# Työmarkkinatori lists each bulletin as a link to a permanent urn.fi address.
BULLETIN_PAGES = [
    "https://tyomarkkinatori.fi/en/employment-and-statistics/evaluation-and-research/employment-bulletin",
    "https://tyomarkkinatori.fi/tyollisyys-ja-tilastot/arviointi-ja-tutkimus/tyollisyyskatsaus",
]
KEHA_URNS = set()
for page in BULLETIN_PAGES:
    page_html = session.get(page, timeout=60).text
    KEHA_URNS.update(re.findall(r"URN:NBN:fi-fe\d+", page_html))
print(len(KEHA_URNS), "KEHA bulletin URNs found")

def read_meta(soup, name):
    tag = soup.find("meta", attrs={"name": name})
    return tag["content"].strip() if tag and tag.get("content") else None

for urn in KEHA_URNS:
    doc_id = "keha_" + urn.split("fi-fe")[-1]
    pdf_path = RAW / f"{doc_id}.pdf"
    if not pdf_path.is_file():
        landing = session.get(f"https://urn.fi/{urn}", headers=HEADERS, timeout=60).url
        soup = BeautifulSoup(session.get(landing, headers=HEADERS, timeout=60).text, "html.parser")
        pdf_url = read_meta(soup, "citation_pdf_url")
        pdf_path.write_bytes(session.get(pdf_url, headers=HEADERS, timeout=60).content)
        title = read_meta(soup, "citation_title")
        language = read_meta(soup, "citation_language")
        published = read_meta(soup, "citation_date")
        time.sleep(1)
    else:
        title, language, published = doc_id, None, None

    reader = PdfReader(str(pdf_path))
    if reader.is_encrypted:
        reader.decrypt("")
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    (TEXT_DIR / f"{doc_id}.txt").write_text(text, encoding="utf-8")

    rows.append({"doc_id": doc_id, "title": title, "language": language,
                "published": published, "source": "keha", "url": f"https://urn.fi/{urn}"})

print(len(KEHA_URNS), "KEHA bulletins saved")

## Ministry bulletins, 2013 to end of 2024

Before KEHA took over, the labour ministry (TEM) published the same bulletin. They are in the
government publication archive: one PDF per month from 2016, and one zip per year for
2013-2015 (12 PDFs inside). We only keep what KEHA does not already cover.

In [ ]:
VN_API = "https://julkaisut.valtioneuvosto.fi/server/api"
FI_MONTHS = ["tammikuu", "helmikuu", "maaliskuu", "huhtikuu", "toukokuu", "kesäkuu",
            "heinäkuu", "elokuu", "syyskuu", "lokakuu", "marraskuu", "joulukuu"]
ZIP_STEMS = {"tammi": 1, "helmi": 2, "maalis": 3, "huhti": 4, "touko": 5, "kesa": 6,
            "heina": 7, "elo": 8, "syys": 9, "loka": 10, "marras": 11, "joulu": 12}

def search_archive(query):
    items = []
    for page in range(5):
        r = session.get(f"{VN_API}/discover/search/objects",
                         params={"query": query, "size": 100, "page": page}, headers=HEADERS)
        objects = r.json()["_embedded"]["searchResult"]["_embedded"]["objects"]
        if not objects:
            break
        items += [o["_embedded"]["indexableObject"] for o in objects]
    return items

def meta(item, key):
    values = item["metadata"].get(key) or [{}]
    return values[0].get("value")

def files_of(uuid):
    item = session.get(f"{VN_API}/core/items/{uuid}", params={"embed": "bundles/bitstreams"}, headers=HEADERS).json()
    out = []
    for bundle in item["_embedded"]["bundles"]["_embedded"]["bundles"]:
        if bundle["name"] == "ORIGINAL":
            out += bundle["_embedded"]["bitstreams"]["_embedded"]["bitstreams"]
    return out

archive_items = search_archive("Työllisyyskatsaus") + search_archive("Työllisyyskatsaukset vuodelta")
print(len(archive_items), "archive items found")

In [ ]:
saved = 0
for item in archive_items:
    title = meta(item, "dc.title") or ""

    monthly = re.match(r"Työllisyyskatsaus, (\w+) (20\d\d)", title)
    if monthly:
        month_name, year = monthly.group(1), int(monthly.group(2))
        if year >= 2025:
            continue  # KEHA already covers this
        month = FI_MONTHS.index(month_name) + 1
        doc_id = f"tem_{year}_{month:02d}"
        pdfs = [f for f in files_of(item["uuid"]) if f["name"].lower().endswith(".pdf")]
        if not pdfs:
            continue
        pdf_path = RAW / f"{doc_id}.pdf"
        if not pdf_path.is_file():
            pdf_path.write_bytes(session.get(pdfs[0]["_links"]["content"]["href"], headers=HEADERS).content)
            time.sleep(1)
        reader = PdfReader(str(pdf_path))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        (TEXT_DIR / f"{doc_id}.txt").write_text(text, encoding="utf-8")
        rows.append({"doc_id": doc_id, "title": title, "language": "fi",
                    "published": f"{year}-{month:02d}-28", "source": "tem", "url": meta(item, "dc.identifier.uri")})
        saved += 1
        continue

    yearly = re.match(r"Työllisyyskatsaukset vuodelta (201[345])", title)
    if yearly:
        year = int(yearly.group(1))
        zips = [f for f in files_of(item["uuid"]) if f["name"].lower().endswith(".zip")]
        if not zips:
            continue
        zip_bytes = session.get(zips[0]["_links"]["content"]["href"], headers=HEADERS).content
        time.sleep(1)
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            for name in zf.namelist():
                stem = next((s for s in ZIP_STEMS if s in name.lower()), None)
                if not stem:
                    continue
                month = ZIP_STEMS[stem]
                doc_id = f"tem_{year}_{month:02d}"
                reader = PdfReader(io.BytesIO(zf.read(name)))
                text = "\n".join(page.extract_text() or "" for page in reader.pages)
                (TEXT_DIR / f"{doc_id}.txt").write_text(text, encoding="utf-8")
                rows.append({"doc_id": doc_id, "title": f"Työllisyyskatsaus {month}/{year}", "language": "fi",
                            "published": f"{year}-{month:02d}-28", "source": "tem", "url": meta(item, "dc.identifier.uri")})
                saved += 1

print(saved, "ministry bulletins saved")

## Save the document list

A simple table with one row per document. This is what the next notebook (chunking and
embedding) reads.

In [ ]:
documents = pd.DataFrame(rows).drop_duplicates("doc_id")
out_dir = REPO / "data" / "processed" / "rag"
out_dir.mkdir(parents=True, exist_ok=True)
documents.to_csv(out_dir / "documents.csv", index=False)

print(len(documents), "documents total")
print(documents.groupby("source").size())